In [ ]:
load_ext jupyter_black

In [ ]:
from copy import deepcopy
import numpy as np
import itertools
import pandas as pd
from tqdm import tqdm
from scipy.stats import pearsonr, spearmanr, ttest_ind_from_stats
import matplotlib.pyplot as plt
import matplotlib.colors as mcolors

In [ ]:
demographics = {
    "prism": [
        "age",
        "gender",
        "employment_status",
        "education",
        "marital_status",
        "english_proficiency",
        "religion",
        "ethnicity",
        "birth_region",
        "reside_region",
        "lm_familiarity",
    ],
    "chen": [
        "Gender",
        "human_Gender",
    ],
    "cad_en": [
        "annotator_age",
        "annotator_gender",
        "annotator_education_level",
        "annotator_political",
        "annotator_ethnicity",
    ],
    "cad_fr": [
        "annotator_age",
        "annotator_gender",
        "annotator_education_level",
        "annotator_political",
        "annotator_ethnicity",
    ],
    "cad_pt": [
        "annotator_age",
        "annotator_gender",
        "annotator_education_level",
        "annotator_political",
        "annotator_ethnicity",
    ],
    "cad_it": [
        "annotator_age",
        "annotator_gender",
        "annotator_education_level",
        "annotator_political",
        "annotator_ethnicity",
    ],
}

domains = ["legal", "salary", "medical", "benefits", "political"]

In [ ]:
questions = pd.read_pickle("data/Llama-3.1-8B-Instruct_questions.gz")
questions_correct_answers = dict(zip(questions.q_id, questions.correct_answer))
domain_qid_map = {
    domain: questions.loc[questions["domain"] == domain, "q_id"].tolist()
    for domain in domains
}

In [ ]:
class WelfordVariance:
    # https://en.wikipedia.org/wiki/Algorithms_for_calculating_variance#Welford's_online_algorithm
    def __init__(self):  # Comparison to ShiftDataVariance:
        self.mean = 0.0  # = K + Ex / n
        self.count = 0  # = n
        self.M2 = 0.0  # = Ex2 - (Ex)^2 / n

    def add_variable(self, x: float):
        self.count += 1
        old_mean = self.mean
        self.mean += (x - self.mean) / self.count
        self.M2 += (x - old_mean) * (x - self.mean)

    def remove_variable(self, x: float):
        self.count -= 1
        new_mean = self.mean
        self.mean -= (x - self.mean) / self.count
        self.M2 -= (x - new_mean) * (x - self.mean)

    def get_mean(self) -> float:
        return self.mean

    def get_variance(self) -> float:
        return self.M2 / self.count

    def get_sample_variance(self) -> float:
        return self.M2 / (self.count - 1)

In [ ]:
from scipy.stats import f as f_dist


def one_way_anova(*groups: tuple[int, float, float]) -> dict:
    """
    Perform a one-way ANOVA from group summary statistics.

    Parameters
    ----------
    *groups : tuples of (n, mean, variance)
        Each tuple describes one group:
          - n   (int)   : number of observations
          - mean (float): sample mean
          - var  (float): sample variance (unbiased, ddof=1)

    Returns
    -------
    dict with keys:
        F        - F-statistic
        p_value  - p-value
        df_between - degrees of freedom between groups
        df_within  - degrees of freedom within groups
        SS_between - sum of squares between groups
        SS_within  - sum of squares within groups
        MS_between - mean square between groups
        MS_within  - mean square within groups
    """
    if len(groups) < 2:
        raise ValueError("At least two groups are required.")

    ns, means, variances = zip(*groups)

    for i, (n, mean, var) in enumerate(groups):
        if n < 2:
            raise ValueError(f"Group {i+1} must have at least 2 samples (n={n}).")
        if var < 0:
            raise ValueError(f"Group {i+1} has a negative variance ({var}).")

    k = len(groups)  # number of groups
    N = sum(ns)  # total observations
    grand_mean = sum(n * m for n, m in zip(ns, means)) / N

    # Between-group sum of squares
    SS_between = sum(n * (m - grand_mean) ** 2 for n, m in zip(ns, means))

    # Within-group sum of squares  (sum of (n-1)*var for each group)
    SS_within = sum((n - 1) * v for n, v in zip(ns, variances))

    df_between = k - 1
    df_within = N - k

    MS_between = SS_between / df_between
    MS_within = SS_within / df_within

    F = MS_between / MS_within
    p_value = f_dist.sf(F, df_between, df_within)  # survival function = 1 - CDF

    return {
        "F": F,
        "p_value": p_value,
        "df_between": df_between,
        "df_within": df_within,
        "SS_between": SS_between,
        "SS_within": SS_within,
        "MS_between": MS_between,
        "MS_within": MS_within,
    }


def print_anova_table(results: dict, alpha: float = 0.05) -> None:
    print("\n" + "=" * 52)
    print("         ONE-WAY ANOVA TABLE")
    print("=" * 52)
    print(f"{'Source':<12} {'SS':>10} {'df':>6} {'MS':>10}")
    print("-" * 52)
    print(
        f"{'Between':<12} {results['SS_between']:>10.4f} {results['df_between']:>6} {results['MS_between']:>10.4f}"
    )
    print(
        f"{'Within':<12} {results['SS_within']:>10.4f} {results['df_within']:>6} {results['MS_within']:>10.4f}"
    )
    print("-" * 52)
    print(f"\n  F-statistic : {results['F']:.4f}")
    print(f"  p-value     : {results['p_value']:.4f}")
    decision = "Reject H₀" if results["p_value"] < alpha else "Fail to reject H₀"
    print(f"  Decision    : {decision}  (α = {alpha})")
    print("=" * 52 + "\n")

In [ ]:
model = "gemma-3-12b-it"  # ["gemma-3-12b-it", "Llama-3.1-8B-Instruct","Qwen3.6-27B"]
dataset = "prism"
domain = "benefits"  # ["benefits", "legal", "medical", "political", "salary"]

In [ ]:
all_cols = deepcopy(demographics[dataset])
if model != "Llama-3.1-8B-Instruct":
    df = pd.read_pickle(f"behavior/{model}_{dataset}_{domain}_answers.gz")
else:
    df = pd.read_pickle(f"behavior/{model}_{dataset}_answers.gz")

for c in domain_qid_map[domain]:
    if domain != "salary":
        df[c] = 1 * (df[c].str.lower() == questions_correct_answers[c])
    else:
        df[c] = df[c].str.replace(",", "").str.extract(r"^[^\d]*(\d+)").astype(float)

df[domain] = df[[qid for qid in domain_qid_map[domain]]].mean(axis=1)
if domain != "salary":
    df[domain] = df[domain] * 100

df = df.drop(
    columns=[f"q_{i}" for i in range(50)]
    + ["q_59", "q_60"]
    + [f"q_{i}" for i in range(61, 211)],
    errors="ignore",
)

df_linguistic = pd.read_pickle(f"data/{dataset}_utterances_linguistic.gz").drop(
    columns=[
        "s_neutral_model_response",
        "s_neutral_user_prompt",
        "model_response_liwc_Segment",
        "user_prompt_liwc_Segment",
    ],
    errors="ignore",
)
for c in ["politeness_user_prompt", "politeness_model_response"]:
    if c in df_linguistic:
        df_linguistic[c] = df_linguistic[c].replace(
            {"impolite": 0, "neutral": 0.5, "polite": 1, "somewhat polite": 0.75}
        )
df_linguistic = df_linguistic.rename(columns={"gpt_description": "topic"})
all_cols += ["topic"]
group_cols = ["conversation_id"] + all_cols
df_linguistic = (
    df_linguistic.groupby(group_cols)[
        [
            c
            for c in df_linguistic.columns
            if ("model_response" in c or "user_prompt" in c)
            and (c not in ["model_response", "user_prompt"])
        ]
    ]
    .mean()
    .reset_index()
)
df = df.merge(
    df_linguistic[
        ["conversation_id"]
        + [
            c
            for c in df_linguistic.columns
            if "model_response" in c or "user_prompt" in c or c == "topic"
        ]
    ],
    on="conversation_id",
)

df = df.loc[df["topic"] != "Outliers"]
results = {}
for column in demographics[dataset]:
    results[column] = {}
    filtered_df = df.loc[
        ~(df[column].isna())
        & (df[column] != "Prefer not to say")
        & (df[column] != "Other")
        & (df[column] != "Unknown")
    ]
    for group in filtered_df[column].unique():
        results[column][group] = {}
        group_df = filtered_df.loc[filtered_df[column] == group]
        for topic in group_df["topic"].unique():
            results[column][group][topic] = group_df.loc[
                group_df["topic"] == topic, domain
            ]

same_group_same_topic = WelfordVariance()
same_group_diff_topic = WelfordVariance()
diff_group_same_topic = WelfordVariance()
diff_group_diff_topic = WelfordVariance()

for column in tqdm(results):
    for group1 in results[column]:
        for group2 in results[column]:
            for topic1 in results[column][group1]:
                for topic2 in results[column][group2]:
                    if group1 == group2 and topic1 == topic2:
                        var = same_group_same_topic
                    elif group1 != group2 and topic1 == topic2:
                        var = diff_group_same_topic
                    elif group1 == group2 and topic1 != topic2:
                        var = same_group_diff_topic
                    else:
                        var = diff_group_diff_topic
                    for e in itertools.product(
                        results[column][group1][topic1],
                        results[column][group2][topic2],
                    ):
                        var.add_variable(abs(e[1] - e[0]))

print(same_group_same_topic.count)
print(model, domain)
print(
    "&$"
    + "$& $".join(
        map(
            "{:.3f}".format,
            [
                same_group_same_topic.get_mean(),
                diff_group_same_topic.get_mean(),
                same_group_diff_topic.get_mean(),
                diff_group_diff_topic.get_mean(),
            ],
        )
    )
    + "$\\\\"
)
print("p: 0.0025")
print(
    ttest_ind_from_stats(
        same_group_same_topic.get_mean(),
        np.sqrt(same_group_same_topic.get_sample_variance()),
        same_group_same_topic.count,
        same_group_diff_topic.get_mean(),
        np.sqrt(same_group_diff_topic.get_sample_variance()),
        same_group_diff_topic.count,
        equal_var=True,
        alternative="two-sided",
    )
)
print(
    ttest_ind_from_stats(
        same_group_same_topic.get_mean(),
        np.sqrt(same_group_same_topic.get_sample_variance()),
        same_group_same_topic.count,
        diff_group_same_topic.get_mean(),
        np.sqrt(diff_group_same_topic.get_sample_variance()),
        diff_group_same_topic.count,
        equal_var=True,
        alternative="two-sided",
    )
)
print(
    ttest_ind_from_stats(
        same_group_diff_topic.get_mean(),
        np.sqrt(same_group_diff_topic.get_sample_variance()),
        same_group_diff_topic.count,
        diff_group_diff_topic.get_mean(),
        np.sqrt(diff_group_diff_topic.get_sample_variance()),
        diff_group_diff_topic.count,
        equal_var=True,
        alternative="two-sided",
    )
)
print(
    ttest_ind_from_stats(
        diff_group_same_topic.get_mean(),
        np.sqrt(diff_group_same_topic.get_sample_variance()),
        diff_group_same_topic.count,
        diff_group_diff_topic.get_mean(),
        np.sqrt(diff_group_diff_topic.get_sample_variance()),
        diff_group_diff_topic.count,
        equal_var=True,
        alternative="two-sided",
    )
)

# print_anova_table(
#     one_way_anova(
#         *[
#             (
#                 same_group_same_topic.count,
#                 same_group_same_topic.get_mean(),
#                 same_group_same_topic.get_variance(),
#             ),
#             (
#                 diff_group_same_topic.count,
#                 diff_group_same_topic.get_mean(),
#                 diff_group_same_topic.get_variance(),
#             ),
#             (
#                 same_group_diff_topic.count,
#                 same_group_diff_topic.get_mean(),
#                 same_group_diff_topic.get_variance(),
#             ),
#             (
#                 diff_group_diff_topic.count,
#                 diff_group_diff_topic.get_mean(),
#                 diff_group_diff_topic.get_variance(),
#             ),
#         ]
#     ),
#     alpha=0.05,
# )

In [ ]:
# ── INPUT: customize each subplot here ────────────────────────────────────────

# x is group, y is topic

plt.rcParams.update({"font.size": 24})

all_subplots = {
    "gemma": [
        {
            "title": "Benefits",
            "values": [
                [5.471, 5.526],
                [6.118, 6.116],
            ],
            "xlabels": ["Same *", "Different *"],
            "ylabels": ["Same *", "Different"],
        },
        {
            "title": "Legal",
            "values": [
                [3.641, 3.649],
                [4.077, 4.057],
            ],
            "xlabels": ["Same *", "Different *"],
            "ylabels": ["Same *", "Different *"],
        },
        {
            "title": "Medical",
            "values": [
                [4.329, 4.364],
                [4.802, 4.792],
            ],
            "xlabels": ["Same *", "Different *"],
            "ylabels": ["Same *", "Different *"],
        },
        {
            "title": "Political",
            "values": [
                [3.157, 3.188],
                [3.378, 3.378],
            ],
            "xlabels": ["Same *", "Different *"],
            "ylabels": ["Same *", "Different"],
        },
        {
            "title": "Salary",
            "values": [
                [1.48321, 1.50751],
                [1.67747, 1.68062],
            ],
            "xlabels": ["Same *", "Different *"],
            "ylabels": ["Same *", "Different *"],
        },
    ],
    "llama": [
        {
            "title": "Benefits",
            "values": [
                [9.255, 9.331],
                [10.515, 10.453],
            ],
            "xlabels": ["Same *", "Different *"],
            "ylabels": ["Same *", "Different *"],
        },
        {
            "title": "Legal",
            "values": [
                [3.583, 3.603],
                [4.245, 4.224],
            ],
            "xlabels": ["Same *", "Different *"],
            "ylabels": ["Same *", "Different *"],
        },
        {
            "title": "Medical",
            "values": [
                [8.445, 8.554],
                [10.119, 10.134],
            ],
            "xlabels": ["Same *", "Different *"],
            "ylabels": ["Same *", "Different *"],
        },
        {
            "title": "Political",
            "values": [
                [3.777, 3.817],
                [4.278, 4.279],
            ],
            "xlabels": ["Same *", "Different *"],
            "ylabels": ["Same *", "Different"],
        },
        {
            "title": "Salary",
            "values": [
                [1.68172, 1.69950],
                [1.87449, 1.87965],
            ],
            "xlabels": ["Same *", "Different *"],
            "ylabels": ["Same *", "Different *"],
        },
    ],
    "qwen": [
        {
            "title": "Benefits",
            "values": [
                [7.932, 8.017],
                [8.661, 8.673],
            ],
            "xlabels": ["Same *", "Different *"],
            "ylabels": ["Same *", "Different *"],
        },
        {
            "title": "Legal",
            "values": [
                [3.041, 3.067],
                [3.319, 3.327],
            ],
            "xlabels": ["Same *", "Different *"],
            "ylabels": ["Same *", "Different *"],
        },
        {
            "title": "Medical",
            "values": [
                [8.667, 8.750],
                [9.456, 9.464],
            ],
            "xlabels": ["Same *", "Different *"],
            "ylabels": ["Same *", "Different *"],
        },
        {
            "title": "Political",
            "values": [
                [2.766, 2.778],
                [2.894, 2.880],
            ],
            "xlabels": ["Same *", "Different *"],
            "ylabels": ["Same *", "Different *"],
        },
        {
            "title": "Salary",
            "values": [
                [1.29988, 1.31710],
                [1.55468, 1.56056],
            ],
            "xlabels": ["Same *", "Different *"],
            "ylabels": ["Same *", "Different *"],
        },
    ],
}

# ── PLOT ──────────────────────────────────────────────────────────────────────

for model in all_subplots:
    subplots = all_subplots[model]
    fig, axes = plt.subplots(1, 5, figsize=(30, 6))
    fig.subplots_adjust(wspace=0.4)

    cmap = "YlOrRd"

    # Compute global vmin/vmax so colours are comparable across subplots
    all_values = [
        v
        for sp in subplots
        for row in sp["values"]
        for v in row
        if sp["title"] != "Salary"
    ]

    vmin, vmax = min(all_values), max(all_values)

    for ax, sp in zip(axes, subplots):
        data = np.array(sp["values"], dtype=float)

        if sp["title"] == "Salary":
            vmin = 1.2
            vmax = 1.9

        im = ax.imshow(data, cmap=cmap, vmin=vmin, vmax=vmax, aspect="equal")

        # Annotate each cell with its numeric value
        for row in range(data.shape[0]):
            for col in range(data.shape[1]):
                val = data[row, col]
                # Choose text colour that contrasts with the cell background
                norm_val = (val - vmin) / (vmax - vmin) if vmax != vmin else 0.5
                text_color = "white" if norm_val > 0.55 else "black"
                ax.text(
                    col,
                    row,
                    f"{val:.3f}",
                    ha="center",
                    va="center",
                    # fontsize=14,
                    fontweight="bold",
                    color=text_color,
                )

        ax.set_title(
            sp["title"].replace("Salary", "Salary (x$1000)"),
            # fontsize=13,
            fontweight="bold",
            pad=10,
        )
        ax.set_xticks([0, 1], sp["xlabels"], fontsize=22)
        ax.set_yticks([0, 1], sp["ylabels"], fontsize=22, rotation=45)
        ax.set_xlabel("Group")
        ax.set_ylabel("Topic")
        ax.tick_params(length=0)

        # Add a per-subplot colourbar
        plt.colorbar(im, ax=ax, fraction=0.046, pad=0.06)

    # fig.suptitle("2×2 Heatmap Grid", fontsize=15, fontweight="bold", y=1.02)
    plt.tight_layout()

    output_path = f"figures_square_behavior/{model}.pdf"
    plt.savefig(output_path, dpi=150, bbox_inches="tight")
    print(f"Saved → {output_path}")
    plt.show()